# 🏥 HỆ THỐNG AI ĐA PHƯƠNG THÁI (2D + 3D CEUS) CHẨN ĐOÁN U BUỒNG TRỨNG
### *Multi-Modal Two-Stage Pipeline: YOLO11 Binary + EfficientNet-B3 on MMOTU Dataset*
---
**Tổng quan Đề tài Nghiên cứu Khoa học:**
- Hợp nhất **cả 2 bộ dữ liệu siêu âm MMOTU**: `OTU_2d` (1.469 ảnh) và `OTU_3d` (170 ảnh CEUS cản âm vi mạch) $\rightarrow$ Tổng cộng: **1.639 ảnh**.
- **Giai đoạn 1 (YOLO11 Binary Detector)**: Sàng lọc đầu vào **Có bệnh (Tumor) vs Bình thường (Normal)** và xác định hộp bao tổn thương (Bounding Box).
- **Giai đoạn 2 (EfficientNet-B3 với SE-Attention)**: Nhận ảnh khối u cắt cận cảnh ($300 \times 300$ px), phân tích hồi âm vi thể để phân loại chính xác **1 trong 7 nhóm bệnh học** và phân tầng nguy cơ ác tính.
- **Báo cáo Đa phương thái**: Xuất bảng đánh giá chi tiết trên ảnh 2D, trên ảnh 3D CEUS và bảng tổng hợp toàn diện.

## 1. Thiết lập Môi trường & Kiểm tra Phần cứng GPU (CUDA)
> **Lưu ý:** Chọn menu **Runtime > Change runtime type > T4 GPU** trước khi bắt đầu.

In [ ]:
!pip install -q ultralytics torchvision timm opencv-python-headless pandas matplotlib seaborn tabulate

import torch
print("=" * 65)
print("🎮 KIỂM TRA PHẦN CỨNG GOOGLE COLAB")
print("=" * 65)
cuda_ok = torch.cuda.is_available()
print(f"CUDA Sẵn sàng: {cuda_ok}")
if cuda_ok:
    print(f"Tên GPU: {torch.cuda.get_device_name(0)}")
    vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"Dung lượng VRAM: {vram:.2f} GB")
else:
    print("⚠️ Cảnh báo: Vui lòng vào Runtime > Change runtime type > T4 GPU!")
print("=" * 65)

## 2. Kết nối Google Drive & Giải nén Dữ liệu MMOTU (2D & 3D)
Nén thư mục `MMOTU` (chứa `OTU_2d` và `OTU_3d`) thành file `MMOTU.zip` và kéo vào Google Drive (`MyDrive/MMOTU.zip`).

In [ ]:
# ==========================================================================
# TỰ ĐỘNG PHÁT HIỆN & NẠP DATASET MMOTU TỪ GOOGLE DRIVE HOẶC GITHUB
# ==========================================================================
from google.colab import drive
import zipfile, os, shutil
from pathlib import Path

# 1. Kết nối Google Drive
drive.mount('/content/drive')

extract_dest = Path('/content/dataset_raw')
extract_dest.mkdir(parents=True, exist_ok=True)

# 2. Các đường dẫn tiềm năng trên Google Drive (bao gồm Drive của tôi > dataset > MMOTU)
drive_candidates = [
    Path('/content/drive/MyDrive/dataset/MMOTU'),
    Path('/content/drive/MyDrive/MMOTU'),
    Path('/content/drive/MyDrive/dataset'),
]

found_folder = None
for p in drive_candidates:
    if p.exists() and ((p / 'OTU_2d').exists() or list(p.glob('**/OTU_2d'))):
        found_folder = p
        break

if found_folder:
    print(f'🎯 Đã tìm thấy dataset trên Google Drive tại: {found_folder}')
    target_2d = extract_dest / 'OTU_2d'
    target_3d = extract_dest / 'OTU_3d'
    if not target_2d.exists():
        print('⚡ Đang sao chép ảnh sang ổ SSD siêu tốc của Colab để huấn luyện nhanh nhất...')
        src_2d = list(found_folder.glob('**/OTU_2d'))[0]
        shutil.copytree(src_2d, target_2d, dirs_exist_ok=True)
        src_3d_list = list(found_folder.glob('**/OTU_3d'))
        if src_3d_list:
            shutil.copytree(src_3d_list[0], target_3d, dirs_exist_ok=True)
    print('✅ Dữ liệu MMOTU (2D + 3D) đã sẵn sàng tại /content/dataset_raw!')

elif os.path.exists('/content/drive/MyDrive/MMOTU.zip'):
    print('📦 Tìm thấy file MMOTU.zip trên Drive. Đang giải nén...')
    with zipfile.ZipFile('/content/drive/MyDrive/MMOTU.zip', 'r') as zip_ref:
        zip_ref.extractall(extract_dest)
    print('✅ Giải nén thành công!')

else:
    print('🚀 Không tìm thấy thư mục trên Drive. Đang tự động clone từ GitHub...')
    !git clone https://github.com/son7nui/NCKH-U-buong-trung.git /content/NCKH
    if (Path('/content/NCKH/MMOTU/OTU_2d')).exists():
        shutil.copytree('/content/NCKH/MMOTU/OTU_2d', extract_dest / 'OTU_2d', dirs_exist_ok=True)
        shutil.copytree('/content/NCKH/MMOTU/OTU_3d', extract_dest / 'OTU_3d', dirs_exist_ok=True)
    print('✅ Đã nạp thành công dữ liệu từ GitHub!')


## 3. Tiền xử lý Đa phương thái: Tự động Ghép Nối 2D + 3D CEUS
- Thêm tiền tố `2d_` và `3d_` để tuyệt đối tránh ghi đè tên file.
- Sinh **Bộ dữ liệu 1**: `dataset_multimodal_binary` (1.639 ảnh cho YOLO11 Binary).
- Sinh **Bộ dữ liệu 2**: `dataset_multimodal_crops_7cls` (các ảnh u cắt $300 \times 300$ px cho EfficientNet-B3).

In [ ]:
import cv2
import numpy as np
from pathlib import Path

base_raw = Path('/content/dataset_raw')

# Tìm thư mục OTU_2d và OTU_3d
dirs_2d = list(base_raw.glob('**/OTU_2d'))
dirs_3d = list(base_raw.glob('**/OTU_3d'))

otu_2d = dirs_2d[0] if dirs_2d else base_raw / 'OTU_2d'
otu_3d = dirs_3d[0] if dirs_3d else base_raw / 'OTU_3d'

print(f"Thư mục 2D: {otu_2d}")
print(f"Thư mục 3D: {otu_3d}")

OUTPUT_BINARY = Path('/content/dataset_multimodal_binary')
OUTPUT_CROPS = Path('/content/dataset_multimodal_crops_7cls')

TUMOR_7_CLASSES = {
    0: 'Chocolate_cyst',
    1: 'Serous_cystadenoma',
    2: 'Teratoma',
    3: 'Theca_cell_tumor',
    4: 'Simple_cyst',
    5: 'Mucinous_cystadenoma',
    6: 'High_grade_serous'
}

ORIGINAL_TO_7CLS = {0: 0, 1: 1, 2: 2, 3: 3, 4: 4, 6: 5, 7: 6}

def load_cls(fpath):
    m = {}
    if not os.path.exists(fpath): return m
    with open(fpath, 'r', encoding='utf-8') as f:
        for l in f:
            p = l.strip().split()
            if len(p) >= 2: m[p[0]] = int(p[1])
    return m

def extract_boxes(mask_path, img_w, img_h, min_area=30):
    if not os.path.exists(mask_path): return [], []
    mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
    if mask is None: return [], []
    thresh = (mask > 0).astype(np.uint8) * 255
    cnts, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    yolo_boxes, pix_boxes = [], []
    for c in cnts:
        if cv2.contourArea(c) < min_area: continue
        x, y, w, h = cv2.boundingRect(c)
        pix_boxes.append((max(0, x), max(0, y), min(img_w, x + w), min(img_h, y + h)))
        xc = min(1.0, max(0.0, (x + w / 2.0) / img_w))
        yc = min(1.0, max(0.0, (y + h / 2.0) / img_h))
        nw = min(1.0, max(0.0, w / float(img_w)))
        nh = min(1.0, max(0.0, h / float(img_h)))
        yolo_boxes.append((xc, yc, nw, nh))
    return yolo_boxes, pix_boxes

def crop_roi(img, box, pad=0.10, sz=(300, 300)):
    h, w = img.shape[:2]
    x1, y1, x2, y2 = box
    bw, bh = x2 - x1, y2 - y1
    cx1 = max(0, x1 - int(bw * pad))
    cy1 = max(0, y1 - int(bh * pad))
    cx2 = min(w, x2 + int(bw * pad))
    cy2 = min(h, y2 + int(bh * pad))
    crop = img[cy1:cy2, cx1:cx2]
    if crop.size == 0: return None
    return cv2.resize(crop, sz, interpolation=cv2.INTER_LINEAR)

def ingest_modality(data_dir, prefix, split, stats):
    cls_map = load_cls(data_dir / f"{split}_cls.txt")
    imgs_dir = data_dir / 'images'
    annots_dir = data_dir / 'annotations'
    bin_img = OUTPUT_BINARY / 'images' / split
    bin_lbl = OUTPUT_BINARY / 'labels' / split
    os.makedirs(bin_img, exist_ok=True)
    os.makedirs(bin_lbl, exist_ok=True)
    
    for img_name, orig_cls in cls_map.items():
        base_id = Path(img_name).stem
        src = imgs_dir / img_name
        if not src.exists(): continue
        img = cv2.imread(str(src))
        if img is None: continue
        h, w = img.shape[:2]
        
        m_path = annots_dir / f"{base_id}_binary.PNG"
        if not m_path.exists(): m_path = annots_dir / f"{base_id}.PNG"
        y_boxes, p_boxes = extract_boxes(m_path, w, h)
        
        # 1. Binary YOLO dataset
        u_name = f"{prefix}_{base_id}"
        shutil.copy2(src, bin_img / f"{u_name}.JPG")
        with open(bin_lbl / f"{u_name}.txt", 'w', encoding='utf-8') as lf:
            if orig_cls == 5:
                stats['normal'] += 1
            else:
                for xc, yc, bw, bh in y_boxes:
                    lf.write(f"0 {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}\n")
                stats['tumor'] += 1
                
        # 2. Crops 7 classes dataset
        if orig_cls != 5 and orig_cls in ORIGINAL_TO_7CLS:
            c_name = TUMOR_7_CLASSES[ORIGINAL_TO_7CLS[orig_cls]]
            c_dir = OUTPUT_CROPS / split / c_name
            os.makedirs(c_dir, exist_ok=True)
            if p_boxes:
                for idx, pbox in enumerate(p_boxes):
                    roi = crop_roi(img, pbox)
                    if roi is not None:
                        cv2.imwrite(str(c_dir / f"{u_name}_crop{idx}.jpg"), roi)
                        stats['crops'] += 1
            else:
                w_roi = cv2.resize(img, (300, 300))
                cv2.imwrite(str(c_dir / f"{u_name}_whole.jpg"), w_roi)
                stats['crops'] += 1

for s in ['train', 'val']:
    st = {'tumor': 0, 'normal': 0, 'crops': 0}
    ingest_modality(otu_2d, '2d', s, st)
    ingest_modality(otu_3d, '3d', s, st)
    print(f"[{s.upper()}] Có u: {st['tumor']} | Bình thường: {st['normal']} | Crops 7 lớp: {st['crops']}")

# Tạo dataset.yaml cho YOLO11
with open(OUTPUT_BINARY / 'dataset.yaml', 'w') as yf:
    yf.write(f"path: {OUTPUT_BINARY}\ntrain: images/train\nval: images/val\nnames:\n  0: Tumor\n")
print("✅ Chuẩn hóa thành công bộ dữ liệu Đa phương thái!")

## 4. Huấn luyện Giai đoạn 1: YOLO11n Binary Detector
Sàng lọc đầu vào Có bệnh vs Bình thường và định vị hộp bao u trên 1.639 ảnh.

In [ ]:
from ultralytics import YOLO

model_stage1 = YOLO('yolo11n.pt')
print("Huấn luyện YOLO11n Binary trên GPU T4 (30 epochs)...")

res_yolo = model_stage1.train(
    data='/content/dataset_multimodal_binary/dataset.yaml',
    epochs=30,
    batch=16,
    imgsz=640,
    device=0,
    optimizer='AdamW',
    lr0=0.001,
    patience=10,
    project='/content/runs_multimodal',
    name='stage1_yolo11_binary',
    plots=True,
    verbose=True
)
print("✅ Giai đoạn 1 hoàn tất! Lưu tại: /content/runs_multimodal/stage1_yolo11_binary/weights/best.pt")

## 5. Huấn luyện Giai đoạn 2: EfficientNet-B3 Phân loại 7 Nhóm U
Nhận diện vi cấu trúc phản âm u $300 \times 300$ px với Class-Weighted CrossEntropy Loss để hỗ trợ u ác tính hiếm (*High-grade serous*).

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.models import efficientnet_b3, EfficientNet_B3_Weights
from collections import Counter

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Data Augmentation cho ảnh siêu âm
train_tf = transforms.Compose([
    transforms.Resize((300, 300)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.15, contrast=0.15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_tf = transforms.Compose([
    transforms.Resize((300, 300)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_ds = datasets.ImageFolder('/content/dataset_multimodal_crops_7cls/train', transform=train_tf)
val_ds = datasets.ImageFolder('/content/dataset_multimodal_crops_7cls/val', transform=val_tf)

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=16, shuffle=False, num_workers=2)

print(f"Tập huấn luyện 7 lớp: {len(train_ds)} ảnh crops")
print(f"Tập kiểm định 7 lớp: {len(val_ds)} ảnh crops")
print(f"Thứ tự các lớp: {train_ds.classes}")

# Tính trọng số Class Weights bù đắp mẫu hiếm
counts = Counter(train_ds.targets)
total_samples = len(train_ds)
class_weights = [total_samples / (len(counts) * counts[i]) for i in range(len(counts))]
weights_tensor = torch.FloatTensor(class_weights).to(device)
print(f"Trọng số Class Weights: {[round(w, 2) for w in class_weights]}")

# Khởi tạo EfficientNet-B3 với pre-trained ImageNet
weights_b3 = EfficientNet_B3_Weights.DEFAULT
model_b3 = efficientnet_b3(weights=weights_b3)
in_f = model_b3.classifier[1].in_features
model_b3.classifier = nn.Sequential(
    nn.Dropout(p=0.3, inplace=True),
    nn.Linear(in_f, 512),
    nn.BatchNorm1d(512),
    nn.SiLU(),
    nn.Dropout(p=0.15),
    nn.Linear(512, 7)
)
model_b3 = model_b3.to(device)

criterion = nn.CrossEntropyLoss(weight=weights_tensor)
optimizer = torch.optim.AdamW(model_b3.parameters(), lr=3e-4, weight_decay=1e-3)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)

EPOCHS_B3 = 20
best_val_acc = 0.0
save_b3_path = '/content/runs_multimodal/efficientnet_b3_best.pth'
os.makedirs('/content/runs_multimodal', exist_ok=True)

print("\nBắt đầu huấn luyện EfficientNet-B3 trên GPU...")
for epoch in range(1, EPOCHS_B3 + 1):
    model_b3.train()
    t_loss, t_correct = 0.0, 0
    for imgs, lbls in train_loader:
        imgs, lbls = imgs.to(device), lbls.to(device)
        optimizer.zero_grad()
        outs = model_b3(imgs)
        loss = criterion(outs, lbls)
        loss.backward()
        optimizer.step()
        t_loss += loss.item() * imgs.size(0)
        t_correct += (outs.argmax(1) == lbls).sum().item()
    scheduler.step()
    
    model_b3.eval()
    v_loss, v_correct = 0.0, 0
    with torch.no_grad():
        for imgs, lbls in val_loader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            outs = model_b3(imgs)
            loss = criterion(outs, lbls)
            v_loss += loss.item() * imgs.size(0)
            v_correct += (outs.argmax(1) == lbls).sum().item()
            
    train_acc = (t_correct / len(train_ds)) * 100
    val_acc = (v_correct / len(val_ds)) * 100
    print(f"Epoch [{epoch:02d}/{EPOCHS_B3}] - Train Acc: {train_acc:.2f}% | Val Acc: {val_acc:.2f}%")
    
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model_b3.state_dict(), save_b3_path)
        
print(f"\n✅ Huấn luyện EfficientNet-B3 hoàn tất! Best Val Acc: {best_val_acc:.2f}%")
print(f"Trọng số lưu tại: {save_b3_path}")

## 6. Đánh giá Đa Phương Thái: Tách Biệt 2D vs 3D CEUS vs Hợp Nhất
Chạy quy trình suy luận 2 giai đoạn (Cascade Pipeline) trên toàn bộ 569 ảnh kiểm định, phân rã kết quả theo:
1. Tập kiểm định siêu âm 2D (469 ảnh)
2. Tập kiểm định siêu âm 3D CEUS cản âm (100 ảnh)
3. Tập kiểm định hợp nhất toàn diện (569 ảnh)

In [ ]:
from PIL import Image
import pandas as pd
from IPython.display import display, HTML

# Load trọng số tốt nhất của cả 2 mô hình
best_yolo = YOLO('/content/runs_multimodal/stage1_yolo11_binary/weights/best.pt')
model_b3.load_state_dict(torch.load(save_b3_path))
model_b3.eval()

class_names = train_ds.classes
CONF_SCREEN = 0.30

val_img_dir = Path('/content/dataset_multimodal_binary/images/val')
val_lbl_dir = Path('/content/dataset_multimodal_binary/labels/val')

results_records = []
all_val_imgs = list(val_img_dir.glob('*.JPG'))

print(f"Chạy suy luận Cascaded Pipeline trên {len(all_val_imgs)} ảnh validation...")

for img_p in all_val_imgs:
    fname = img_p.name
    modality = '2D' if fname.startswith('2d_') else '3D_CEUS'
    base_name = img_p.stem
    
    # Ground Truth nhị phân
    lbl_p = val_lbl_dir / f"{base_name}.txt"
    is_gt_tumor = False
    if lbl_p.exists():
        with open(lbl_p, 'r') as lf:
            if len(lf.read().strip()) > 0: is_gt_tumor = True
            
    # GIAI ĐOẠN 1: YOLO11 Sàng lọc
    y_preds = best_yolo.predict(source=str(img_p), conf=CONF_SCREEN, verbose=False)
    has_detected_box = len(y_preds) > 0 and len(y_preds[0].boxes) > 0
    
    pred_disease = False
    pred_class_name = 'Normal_Ovary'
    pred_conf = 0.0
    risk_level = 'Bình thường'
    
    if has_detected_box:
        pred_disease = True
        # GIAI ĐOẠN 2: Cắt ROI và đưa qua EfficientNet-B3
        box = y_preds[0].boxes.xyxy[0].cpu().numpy().astype(int)
        img_cv = cv2.imread(str(img_p))
        roi = crop_roi(img_cv, box)
        if roi is not None:
            roi_rgb = cv2.cvtColor(roi, cv2.COLOR_BGR2RGB)
            pil_img = Image.fromarray(roi_rgb)
            inp = val_tf(pil_img).unsqueeze(0).to(device)
            with torch.no_grad():
                logits = model_b3(inp)
                probs = torch.softmax(logits, dim=1)[0]
                top_idx = probs.argmax().item()
                pred_class_name = class_names[top_idx]
                pred_conf = probs[top_idx].item() * 100
                
            # Phân tầng nguy cơ ác tính
            if pred_class_name == 'High_grade_serous' or probs[-1].item() >= 0.30:
                risk_level = 'Ác tính (High-grade)'
            else:
                risk_level = 'Lành tính / U nang'
                
    results_records.append({
        'filename': fname,
        'modality': modality,
        'gt_tumor': is_gt_tumor,
        'pred_tumor': pred_disease,
        'pred_class': pred_class_name,
        'confidence': pred_conf,
        'risk_level': risk_level
    })

df_res = pd.DataFrame(results_records)

# Hàm tính toán chỉ số theo từng nhóm
def calc_stats(sub_df, name):
    tp = len(sub_df[(sub_df['gt_tumor'] == True) & (sub_df['pred_tumor'] == True)])
    fp = len(sub_df[(sub_df['gt_tumor'] == False) & (sub_df['pred_tumor'] == True)])
    tn = len(sub_df[(sub_df['gt_tumor'] == False) & (sub_df['pred_tumor'] == False)])
    fn = len(sub_df[(sub_df['gt_tumor'] == True) & (sub_df['pred_tumor'] == False)])
    total = len(sub_df)
    sens = (tp / (tp + fn)) * 100 if (tp + fn) > 0 else 0
    spec = (tn / (tn + fp)) * 100 if (tn + fp) > 0 else 0
    acc = ((tp + tn) / total) * 100 if total > 0 else 0
    return {
        'Bộ Dữ Liệu': name,
        'Số lượng Mẫu': total,
        'Đoán Đúng U (TP)': tp,
        'Báo Động Giả (FP)': fp,
        'Chuẩn Âm Tính (TN)': tn,
        'Bỏ Sót U (FN)': fn,
        'Độ Nhạy (Sensitivity)': f"{sens:.2f}%",
        'Độ Đặc Hiệu (Specificity)': f"{spec:.2f}%",
        'Độ Chính Xác Sàng Lọc': f"{acc:.2f}% ⭐"
    }

summary_data = [
    calc_stats(df_res[df_res['modality'] == '2D'], 'Siêu âm 2D Thông thường'),
    calc_stats(df_res[df_res['modality'] == '3D_CEUS'], 'Siêu âm 3D CEUS Cản âm vi mạch'),
    calc_stats(df_res, 'TỔNG HỢP TOÀN BỘ (2D + 3D CEUS)')
]
df_summary = pd.DataFrame(summary_data)

# Render HTML
html_markup = f"""
<div style='font-family: Arial, sans-serif; max-width: 950px; margin: 20px 0;'>
  <h3 style='color: #0284c7; border-bottom: 2px solid #0284c7; padding-bottom: 8px;'>🏆 BẢNG TỔNG HỢP HIỆU NĂNG ĐA PHƯƠNG THÁI (2D vs 3D CEUS vs HỢP NHẤT)</h3>
  {df_summary.to_html(index=False, classes='table table-bordered table-striped')}
</div>
"""
display(HTML(html_markup))

df_summary.to_csv('/content/runs_multimodal/multimodal_summary_report.csv', index=False, encoding='utf-8-sig')
print("Đã lưu báo cáo thành: /content/runs_multimodal/multimodal_summary_report.csv")

## 7. Trực quan Hóa Ca Bệnh & Phân Tầng Rủi Ro
Hiển thị hình ảnh đối chiếu thực tế gồm: Bounding Box khoanh vùng, Tên loại u chẩn đoán bởi EfficientNet-B3, và Nhãn cảnh báo rủi ro màu đỏ/xanh.

In [ ]:
import matplotlib.pyplot as plt

sample_test_imgs = all_val_imgs[:3]
fig, axes = plt.subplots(1, len(sample_test_imgs), figsize=(16, 5))

for i, p in enumerate(sample_test_imgs):
    row = df_res[df_res['filename'] == p.name].iloc[0]
    img_bgr = cv2.imread(str(p))
    y_preds = best_yolo.predict(source=str(p), conf=CONF_SCREEN, verbose=False)
    
    if len(y_preds) > 0 and len(y_preds[0].boxes) > 0:
        box = y_preds[0].boxes.xyxy[0].cpu().numpy().astype(int)
        color = (0, 0, 255) if 'Ác tính' in row['risk_level'] else (0, 255, 0)
        cv2.rectangle(img_bgr, (box[0], box[1]), (box[2], box[3]), color, 2)
        txt = f"{row['pred_class']} ({row['confidence']:.1f}%)"
        cv2.putText(img_bgr, txt, (box[0], max(20, box[1] - 8)), cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2)
        
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    axes[i].imshow(img_rgb)
    axes[i].axis('off')
    axes[i].set_title(f"[{row['modality']}] {row['pred_class']}\n{row['risk_level']}", fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

## 8. Tự động Sao lưu Toàn bộ Trọng số & Kết quả về Google Drive
Lưu các file quan trọng vào thư mục `MyDrive/NCKH_Multimodal_Results/`.

In [ ]:
backup_dir = Path('/content/drive/MyDrive/NCKH_Multimodal_Results')
os.makedirs(backup_dir, exist_ok=True)

# Sao lưu Model 1 YOLO11
yolo_best = '/content/runs_multimodal/stage1_yolo11_binary/weights/best.pt'
if os.path.exists(yolo_best):
    shutil.copy2(yolo_best, backup_dir / 'stage1_yolo11_binary_multimodal_best.pt')
    print(f"✅ Đã sao lưu YOLO11 Binary: {backup_dir}/stage1_yolo11_binary_multimodal_best.pt")

# Sao lưu Model 2 EfficientNet-B3
if os.path.exists(save_b3_path):
    shutil.copy2(save_b3_path, backup_dir / 'stage2_efficientnet_b3_multimodal_best.pth')
    print(f"✅ Đã sao lưu EfficientNet-B3: {backup_dir}/stage2_efficientnet_b3_multimodal_best.pth")

# Sao lưu bảng kết quả tổng hợp
sum_csv = '/content/runs_multimodal/multimodal_summary_report.csv'
if os.path.exists(sum_csv):
    shutil.copy2(sum_csv, backup_dir / 'multimodal_summary_report.csv')
    print(f"✅ Đã sao lưu Báo cáo CSV: {backup_dir}/multimodal_summary_report.csv")

print(f"\n🎉 HOÀN TẤT! Tất cả dữ liệu nghiên cứu đã được bảo toàn an toàn trên Google Drive!")